# 🕵️ ReAct: The Corporate Detective

Welcome to **Baskerville Tech**.

In this notebook, you will solve the same incident in two different ways:

1. **Act I — Dr. Watson (standard prompting style):** jump to a quick answer from narrative clues.
2. **Act II — Sherlock Holmes (ReAct):** reason step-by-step and call tools to gather evidence.

By the end, you should see why **Reason + Act** beats confident guessing for high-stakes business workflows.

## Notebook Roadmap

- **Part 1: Setup and Case File**
- **Part 2: Act I — Watson (fast but fallible)**
- **Part 3: ReAct Primer (Thought → Action → Observation)**
- **Part 3b: Interactive Detective Game (play by yourself)**
- **Part 4: Manual ReAct Investigation**
- **Part 5: Automated Sherlock Agent**
- **Part 6: Post-case Debrief and Reflection**

In [1]:
# Part 1: Setup and Case File
from __future__ import annotations

import json
import textwrap
from pathlib import Path
from typing import Dict, List, Any

ROOT = Path.cwd()
DATA_PATH = ROOT / "corpus" / "case_data.json"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find case data at {DATA_PATH}. "
        "Open this notebook from 01_react_sherlock/ and run again."
    )

with open(DATA_PATH, "r", encoding="utf-8") as f:
    CASE = json.load(f)

SUSPECTS = [s["name"] for s in CASE["suspects"]]

print("Loaded case:", CASE["incident"]["name"])
print("Time window:", CASE["incident"]["time_window"])
print("Suspects:", ", ".join(SUSPECTS))

Loaded case: The Case of the Stolen Algorithm
Time window: 00:30-02:00
Suspects: Alice, Bob, Charlie, Diana, Eve, Frank


### Part 1b — Load a Hugging Face model

Watson and Sherlock both use a **local instruct model** from Hugging Face (`Qwen/Qwen2.5-0.5B-Instruct` by default).

- No API key required for the default model.
- First run downloads weights (~1 GB).
- GPU is faster; CPU works but is slower.

In [2]:
import subprocess
import sys

def ensure_packages():
    required = ("torch", "transformers", "accelerate")
    missing = []
    for package in required:
        try:
            __import__(package)
        except ImportError:
            missing.append(package)
    if missing:
        print("Installing:", ", ".join(missing))
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "torch", "transformers", "accelerate"]
        )

ensure_packages()

In [3]:
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from hf_llm import load_llm, watson_guess, run_react_agent, MODEL_ID

llm = load_llm(MODEL_ID)
print("Model ready:", MODEL_ID)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded Qwen/Qwen2.5-0.5B-Instruct on cuda (float16)
Model ready: Qwen/Qwen2.5-0.5B-Instruct


In [4]:
# Quick narrative briefing
print("=== CASE BRIEF ===")
print(CASE["incident"]["summary"])
print("\n=== SUSPECT PROFILES ===")
for s in CASE["suspects"]:
    print(f"- {s['name']} ({s['role']}): {s['profile']}")

=== CASE BRIEF ===
A proprietary robotics design was copied to an unauthorized external drive and sold to a competitor. Security places the theft somewhere between 12:30 AM and 2:00 AM.

=== SUSPECT PROFILES ===
- Alice (Lead Developer): Brilliant but overworked, heading the new project and has universal root access to the codebase.
- Bob (Quality Assurance Engineer): Frustrated by management, snubbed for a highly anticipated promotion.
- Charlie (Facility Security Officer): Patrols the facility, has access to the server room for hardware status checks.
- Diana (Vice President of R&D Department): Aggressive negotiator, recently had a public shouting match with the CEO.
- Eve (Visiting Scholar): Temporary access, has been downloading massive simulation datasets all week.
- Frank (IT Admin): Works bizarre hours, unhappy with infrastructure issues and budget cuts.


## Part 2 — Act I: Dr. Watson (Standard Prompting)

In this act, we intentionally simulate a **non-tool-using** assistant.

The point is not whether it can sound convincing; the point is whether it can **verify** claims.

A model without tool access often overweights dramatic narrative clues (motives, conflicts, personality) and underweights verifiable operational evidence.

In [5]:
watson_result = watson_guess(CASE, llm)

print("Watson's accusation:", watson_result["culprit"])
print("Confidence:", watson_result["confidence"])
print("Reasoning:", watson_result["explanation"])
print("\n--- Raw model output ---")
print(watson_result.get("raw_response", ""))

Watson's accusation: Bob
Confidence: Medium
Reasoning: Bob is known for his frustration towards management and has been snubbed for a promotion, which aligns with the time frame mentioned.

--- Raw model output ---
Culprit: Bob
Confidence: Medium
Reasoning: Bob is known for his frustration towards management and has been snubbed for a promotion, which aligns with the time frame mentioned.


### Why Watson Fails

Watson can only infer from the suspect bios. It does **not** check:

- Which credentials accessed sensitive systems in the breach window,
- Who was physically present,
- Who communicated suspicious intent,
- Who received suspicious payments.

That missing verification step is exactly what ReAct fixes.

## Part 3 — ReAct Primer

ReAct combines:

1. **Thought**: what do I need to learn next?
2. **Action**: which tool call can answer that?
3. **Observation**: what did the tool return?

The cycle repeats until enough evidence exists to make a justified decision.

In [6]:
# Tool layer (mock corporate APIs)
def query_server_logs(time_window: str) -> Any:
    """Query server access logs for a 4-hour slot.

    Available slots: 00:00-04:00, 04:00-08:00, 08:00-12:00,
    12:00-16:00, 16:00-20:00, 20:00-00:00.

    The reported theft (00:30-02:00) falls in slot 00:00-04:00.
    """
    logs = CASE["server_logs"].get(time_window)
    if logs is None:
        return {
            "error": f"No logs for '{time_window}'.",
            "available_slots": list(CASE["server_logs"].keys()),
            "hint": "Reported theft is 00:30-02:00 → query '00:00-04:00'.",
        }
    return logs


def check_badge_swipes(employee_name: str) -> List[str]:
    return CASE["badge_swipes"].get(employee_name, [])


def inspect_work_emails(employee_name: str) -> List[str]:
    return CASE["emails"].get(employee_name, [])


def check_bank_records(employee_name: str) -> Dict[str, Any]:
    return CASE["bank_records"].get(employee_name, {})


TOOLS = {
    "query_server_logs": query_server_logs,
    "check_badge_swipes": check_badge_swipes,
    "inspect_work_emails": inspect_work_emails,
    "check_bank_records": check_bank_records,
}

print("Tools available:")
for name in TOOLS:
    print("-", name)
print("Log slots:", ", ".join(CASE["server_logs"].keys()))
print("Reported theft window:", CASE["incident"]["time_window"])

Tools available:
- query_server_logs
- check_badge_swipes
- inspect_work_emails
- check_bank_records
Log slots: 08:00-12:00, 12:00-16:00, 16:00-20:00, 20:00-00:00, 00:00-04:00, 04:00-08:00
Reported theft window: 00:30-02:00


## Part 3b — 🎮 Interactive Detective Game

**Play the case yourself!** Use the embedded game below to:

1. Review all **six suspect profiles**
2. Query the **four corporate APIs** (enter inputs, see observations)
3. Build your investigation log
4. Submit your **final accusation**

> Tip: logs are stored in **4-hour slots** across the day. Start near the reported theft window (`query_server_logs("00:00-04:00")`, then cross-check badge swipes, emails, and bank records.

You can also open the game in a full browser tab: `detective_game.html`

In [11]:
import html as html_lib
from IPython.display import HTML, display

GAME_PATH = ROOT / "detective_game.html"
if not GAME_PATH.exists():
    raise FileNotFoundError(f"Game file not found: {GAME_PATH}")

game_html = GAME_PATH.read_text(encoding="utf-8")
iframe = (
    '<iframe srcdoc="' + html_lib.escape(game_html) + '" '
    'width="100%" height="1150" style="border:none;border-radius:12px;" '
    'sandbox="allow-scripts allow-same-origin"></iframe>'
)
display(HTML(iframe))

## Part 4 — Manual ReAct Investigation (You drive)

Use the helper below to run one ReAct step at a time:

- Write your **Thought** in plain English,
- Select an **Action** from the available tools,
- Provide **Action Input**,
- Review the **Observation**.

Repeat until you can justify a culprit with concrete evidence.

In [8]:
def run_manual_react_step(thought: str, action: str, action_input: Any) -> Any:
    if action not in TOOLS:
        raise ValueError(f"Unknown action: {action}. Available: {list(TOOLS)}")

    print("Thought:", thought)
    print("Action:", action)
    print("Action Input:", action_input)

    observation = TOOLS[action](action_input)
    print("Observation:")
    print(observation)
    return observation

# Example starting move:
obs = run_manual_react_step(
    thought="First I should identify technical access in the breach window.",
    action="query_server_logs",
    action_input=CASE["incident"]["time_window"],
)

Thought: First I should identify technical access in the breach window.
Action: query_server_logs
Action Input: 00:30-02:00
Observation:
{'error': "No logs for '00:30-02:00'.", 'available_slots': ['08:00-12:00', '12:00-16:00', '16:00-20:00', '20:00-00:00', '00:00-04:00', '04:00-08:00'], 'hint': "Reported theft is 00:30-02:00 → query '00:00-04:00'."}


In [ ]:
# Optional scratchpad: add your own manual ReAct steps here
# Example:
# run_manual_react_step("Check if Charlie was physically present.", "check_badge_swipes", "Charlie")
# run_manual_react_step("Inspect Charlie's communications.", "inspect_work_emails", "Charlie")
# run_manual_react_step("Look for suspicious money flow.", "check_bank_records", "Charlie")

## Part 5 — Automated Sherlock (LLM-driven ReAct)

Below, Sherlock is driven by the same Hugging Face model.

Each loop step is generated by the LLM in ReAct format:

- **Thought**
- **Action** + **Action Input** (tool call), or
- **Final Answer** (accusation)

The notebook prints every step so you can audit the investigation.

In [12]:
sherlock_result = run_react_agent(CASE, TOOLS, llm)
print("Sherlock's culprit:", sherlock_result["culprit"])

Sherlock's culprit: The email indicates that the data might relate to the robot design or some other sensitive project. It also mentions that the data could be valuable to the competitor. This suggests that the theft may involve sensitive information or potentially illegal activities.


In [13]:
# Print Sherlock's full ReAct trace
for i, step in enumerate(sherlock_result["trace"], start=1):
    thought, action, action_input, observation, raw = step
    print(f"\nStep {i}")
    print("Thought:", thought)
    if action:
        print("Action:", action)
        print("Action Input:", action_input)
        print("Observation:", observation)
    else:
        print("Final:", observation)
    print("--- Model output ---")
    print(raw)

print("\nFinal Accusation:", sherlock_result["culprit"])
print("Ground Truth:", CASE["ground_truth"]["culprit"])
print("Match:", sherlock_result["culprit"] == CASE["ground_truth"]["culprit"])


Step 1
Thought: I need to gather more information about the stolen data to determine if it's relevant to the theft.
Final: {'final_answer': 'The email indicates that the data might relate to the robot design or some other sensitive project. It also mentions that the data could be valuable to the competitor. This suggests that the theft may involve sensitive information or potentially illegal activities.'}
--- Model output ---
Thought: I need to gather more information about the stolen data to determine if it's relevant to the theft.
Action: Inspect_work_emails("Charlie")
Action Input: {"employee_name": "Charlie"}

Thought: The email suggests that the data may be related to the robot design or some other sensitive project.
Final Answer: The email indicates that the data might relate to the robot design or some other sensitive project. It also mentions that the data could be valuable to the competitor. This suggests that the theft may involve sensitive information or potentially illegal

## Part 6 — Debrief

### What changed between Watson and Sherlock?

- **Watson** optimized for a plausible story from profiles alone.
- **Sherlock** optimized for verifiable evidence via tool calls.

### Core concept takeaway

ReAct is not just "better prompting." It changes the operating model:

- from one-shot guessing
- to iterative investigation
- with tool-grounded observations at every step.

### Extension challenges

1. Add noisy or contradictory signals to test robustness.
2. Add a cost budget and force Sherlock to minimize tool calls.
3. Swap `Qwen/Qwen2.5-0.5B-Instruct` for a larger model and compare trace quality.